# Data Processing

In [1]:
# import libraries
import numpy as np
import pandas as pd

# import data
raw = pd.read_excel("/Users/jiehni/monash/ADS2001_GROUPMC3/data/raw/Melbourne.xlsx")
raw.head()

,Year,Month,Day,Hour,Min,Air Temp (degrees C),Apparent Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Direction,Wind Speed (km/h),Wind Gust (km/h),MSLP (hPa),Rainfall since 9 am (mm),gamma,Calculated Dew Pt Temp (degrees C),E (hPa),Calculated Apparent Temp (degrees C)
0,2011,1,1.0,0.0,4.0,24.8,0.0,14.0,51.0,SE,11,13.0,1007.4,0,0.969609,14.1,15.916676,23.9
1,2011,1,1.0,0.0,14.0,24.8,0.0,13.3,48.0,SE,11,11.0,1007.5,0,0.908985,13.2,14.980401,23.6
2,2011,1,1.0,0.0,24.0,24.9,0.0,13.3,48.0,SE,11,13.0,1007.5,0,0.915025,13.2,15.069879,23.7
3,2011,1,1.0,0.0,34.0,24.7,0.0,13.4,49.0,SE,11,11.0,1007.4,0,0.923560,13.4,15.201624,23.6
4,2011,1,1.0,0.0,44.0,24.1,0.0,13.3,51.0,ESE,9,9.0,1007.3,0,0.927209,13.4,15.264860,23.4


In [2]:
# create target variable for rain (y/n)
raw["Rainfall since 9 am (mm)"] = pd.to_numeric(raw["Rainfall since 9 am (mm)"], errors="coerce")
raw["Rain"] = (raw["Rainfall since 9 am (mm)"] > 0).astype(int)

In [3]:
# missing values
(raw.isna().sum()/len(raw))*100

Year                                    0.026318
Month                                   0.121895
Day                                     0.125061
Hour                                    0.125061
Min                                     0.125061
Air Temp (degrees C)                    0.125061
Apparent Temp (degrees C)               0.125061
Dew Pt Temp (degrees C)                 0.125061
Humidity (%)                            0.125061
Wind Direction                          0.125061
Wind Speed (km/h)                       0.125061
Wind Gust  (km/h)                       0.125061
MSLP (hPa)                              0.125061
Rainfall since 9 am (mm)                0.140891
gamma                                   0.127633
Calculated Dew Pt Temp (degrees C)      0.127633
E (hPa)                                 0.000000
Calculated Apparent Temp (degrees C)    0.026318
Rain                                    0.000000
dtype: float64

As missing values make up less than 1% of values, we decide to drop them

In [4]:
# drop empty values
raw = raw.dropna()
(raw.isna().sum()/len(raw))*100

Year                                    0.0
Month                                   0.0
Day                                     0.0
Hour                                    0.0
Min                                     0.0
Air Temp (degrees C)                    0.0
Apparent Temp (degrees C)               0.0
Dew Pt Temp (degrees C)                 0.0
Humidity (%)                            0.0
Wind Direction                          0.0
Wind Speed (km/h)                       0.0
Wind Gust  (km/h)                       0.0
MSLP (hPa)                              0.0
Rainfall since 9 am (mm)                0.0
gamma                                   0.0
Calculated Dew Pt Temp (degrees C)      0.0
E (hPa)                                 0.0
Calculated Apparent Temp (degrees C)    0.0
Rain                                    0.0
dtype: float64

In [5]:
raw["Datetime"] = pd.to_datetime({
    "year": raw["Year"],
    "month": raw["Month"],
    "day": raw["Day"],
    "hour": raw["Hour"],
    "minute": raw["Min"],
}, errors="coerce"
)
raw["Datetime"] = pd.to_datetime(raw["Datetime"])

# drop original variables
raw = raw.drop(columns=["Year", "Month", "Day", "Hour", "Min"])

# create new datetime variables
raw["day_of_year"] = raw["Datetime"].dt.dayofyear
raw["hour"] = raw["Datetime"].dt.hour
raw["month"] = raw["Datetime"].dt.month
raw["day_of_week"] = raw["Datetime"].dt.dayofweek
raw.head()

,Air Temp (degrees C),Apparent Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Direction,Wind Speed (km/h),Wind Gust (km/h),MSLP (hPa),Rainfall since 9 am (mm),gamma,Calculated Dew Pt Temp (degrees C),E (hPa),Calculated Apparent Temp (degrees C),Rain,Datetime,day_of_year,hour,month,day_of_week
0,24.8,0.0,14.0,51.0,SE,11,13.0,1007.4,0.0,0.969609,14.1,15.916676,23.9,0,2011-01-01 00:04:00,1,0,1,5
1,24.8,0.0,13.3,48.0,SE,11,11.0,1007.5,0.0,0.908985,13.2,14.980401,23.6,0,2011-01-01 00:14:00,1,0,1,5
2,24.9,0.0,13.3,48.0,SE,11,13.0,1007.5,0.0,0.915025,13.2,15.069879,23.7,0,2011-01-01 00:24:00,1,0,1,5
3,24.7,0.0,13.4,49.0,SE,11,11.0,1007.4,0.0,0.923560,13.4,15.201624,23.6,0,2011-01-01 00:34:00,1,0,1,5
4,24.1,0.0,13.3,51.0,ESE,9,9.0,1007.3,0.0,0.927209,13.4,15.264860,23.4,0,2011-01-01 00:44:00,1,0,1,5


In [6]:
# check duplicates
raw.duplicated().sum()

0

In [9]:
# encode wind direction (circular)
wind_dir_map = {
    "N":0, 
    "NNE":22.5, 
    "NE":45, 
    "ENE":67.5,
    "E":90, 
    "ESE":112.5, 
    "SE":135, 
    "SSE":157.5,
    "S":180, 
    "SSW":202.5, 
    "SW":225, 
    "WSW":247.5,
    "W":270, 
    "WNW":292.5, 
    "NW":315, 
    "NNW":337.5
}

raw["wind_angle"] = raw["Wind Direction"].map(wind_dir_map) # convert to wind angle
raw["wind_sin"] = np.sin(np.deg2rad(raw["wind_angle"]))
raw["wind_cos"] = np.cos(np.deg2rad(raw["wind_angle"]))

# drop columns
raw = raw.drop(columns=["Wind Direction", "wind_angle"])

raw.head()

,Air Temp (degrees C),Apparent Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),Wind Gust (km/h),MSLP (hPa),Rainfall since 9 am (mm),gamma,Calculated Dew Pt Temp (degrees C),E (hPa),Calculated Apparent Temp (degrees C),Rain,Datetime,day_of_year,hour,month,day_of_week,wind_sin,wind_cos
0,24.8,0.0,14.0,51.0,11,13.0,1007.4,0.0,0.969609,14.1,15.916676,23.9,0,2011-01-01 00:04:00,1,0,1,5,0.707107,-0.707107
1,24.8,0.0,13.3,48.0,11,11.0,1007.5,0.0,0.908985,13.2,14.980401,23.6,0,2011-01-01 00:14:00,1,0,1,5,0.707107,-0.707107
2,24.9,0.0,13.3,48.0,11,13.0,1007.5,0.0,0.915025,13.2,15.069879,23.7,0,2011-01-01 00:24:00,1,0,1,5,0.707107,-0.707107
3,24.7,0.0,13.4,49.0,11,11.0,1007.4,0.0,0.923560,13.4,15.201624,23.6,0,2011-01-01 00:34:00,1,0,1,5,0.707107,-0.707107
4,24.1,0.0,13.3,51.0,9,9.0,1007.3,0.0,0.927209,13.4,15.264860,23.4,0,2011-01-01 00:44:00,1,0,1,5,0.923880,-0.382683


In [55]:
# save processed data
raw.to_csv("/Users/jiehni/monash/ADS2001_GROUPMC3/data/processed/melb_processed.csv", index=False)

In [12]:
print(raw["Rainfall since 9 am (mm)"].min())

0.0
